# Step 1 (Bias Correction) Evaluation Notebook

This notebook is the **research artifact used to produce the Step 1 results
reported in the paper**. It loads multiple trained checkpoints (some used as
the final model, others as baselines or ablations) and computes the evaluation
metrics shown in Tables X-Y.

## Reproducing this notebook

This notebook expects:

1. **Trained checkpoints** in `../output/` with the original training-run
   directory names. To reproduce the paper's results, either:
   - To use the checkpoints from the paper, contact the corresponding author for access (see README). Otherwise, retrain using the configs in ../conf/ and update the paths below to point to your local checkpoint files.
   - Or retrain models yourself using the configs in `../conf/`

2. **Bias-corrected validation outputs** in `../data/` (the
   `AIFS_bias_corrected_precip_VAL_*.nc` files). These are produced by
   running the trained Step 1 model on the validation set; see
   `scripts/run_step1_inference.py` for the generation script.

⚠️ **This notebook is provided for reproducibility and reference**, not as
a tutorial. The model paths in Cell 18 reflect the directory structure of
the original training runs and will likely require adjustment.

In [ ]:
import json
import os
import sys
import pickle

import numpy as np
import pandas as pd
import xarray as xr
import torch
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
from torch.utils.data import DataLoader, Subset

from physicsnemo.models.diffusion import UNet, EDMPrecondSuperResolution
from physicsnemo.utils.diffusion import InfiniteSampler
from physicsnemo.distributed import DistributedManager
from physicsnemo import Module

plt.style.use("ggplot")
plt.rcParams["lines.linewidth"] = 2

In [ ]:
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from models.unet_corrector import UNetResidualCorrector
from models.unet_corrector_clamped import UNetResidualCorrectorClamped
from models import ViTBiasCorrectorFiLM
from datasets.dataset import init_dataset_from_config, register_dataset

from helpers.viz_helpers import ColormapConfig
from helpers.bc_metrics import (
    baseline_per_lead_metrics,
    EvaluateModel,
    EvaluateModelPerLead,
)
from helpers.viz_helpers import Visualize_Pred_onebatch_ByLeadTime3
from helpers.bc_plots import (
    plot_metric_comparison_grouped_legend,
    plot_fss_by_lead,
    plot_sal_by_lead,
)
from helpers.bc_tables import (
    generate_latex_tables,
    generate_compact_latex_table,
    generate_per_lead_latex_table,
)
from helpers.bc_inference import (
    denorm_and_delog,
    RunInferenceWeekly,
    RunInferenceWeeklySingleLead,
)

from models.baselines_bias_correction import (
    build_baseline_spatiotemporal_from_dataset,
    BaselineB0_Raw,
    BaselineB1_AffineGlobal,
    BaselineB2_AffinePerLeadGlobal,
    BaselineB3_AffinePerGrid,
    BaselineB4_MultiLinearPerGrid,
    BaselineB5_QuantileMapping,
    BaselineB7_RidgeGlobal,
    BaselineB8_RFGlobal,
)

In [ ]:
import cartopy.crs as ccrs
import cartopy.io.shapereader as shpreader

shp_fname = shpreader.natural_earth(
    resolution="50m", category="cultural", name="admin_0_countries"
)
reader = shpreader.Reader(shp_fname)
switz_geom = next(
    rec.geometry
    for rec in reader.records()
    if rec.attributes["NAME_LONG"] == "Switzerland"
)

In [ ]:
DATA_ROOT = "../data"
CONF_ROOT = "../conf"
OUT_ROOT = "../output"
RESULTS_ROOT = "../output/images_results"
os.makedirs(RESULTS_ROOT, exist_ok=True)

## 1) Dataset initialisation and weekly-split data loaders

In [ ]:
DATA_LOADER_KWARGS = {"pin_memory": True, "num_workers": 0, "prefetch_factor": None}


def init_data(cfg):
    """Initialise DDP and instantiate the dataset from a Hydra config."""
    DistributedManager.initialize()
    dataset_cfg = OmegaConf.to_container(cfg.dataset)
    register_dataset(cfg.dataset.type)
    full_dataset, _ = init_dataset_from_config(
        dataset_cfg,
        DATA_LOADER_KWARGS,
        None,
        0,
    )
    print(f"Dataset length: {len(full_dataset)}")
    return full_dataset


def build_weekly_split_loaders(full_dataset, batch_size=256):
    """Build 6-on / 1-off weekly train/val split and return loaders."""
    N = len(full_dataset)
    times_list = full_dataset.time()

    # Infer number of leads from consecutive identical timestamps
    first_time = times_list[0]
    L = 1
    for i in range(1, N):
        if times_list[i] == first_time:
            L += 1
        else:
            break

    T_unique = N // L
    print(f"Leads L={L}, unique times={T_unique}")

    train_idx, val_idx = [], []
    for g in range(T_unique):
        start, end = g * L, (g + 1) * L
        (train_idx if (g % 7) < 6 else val_idx).extend(range(start, end))

    print(f"Train: {len(train_idx)}, Val: {len(val_idx)}")
    assert train_idx and val_idx, "Empty split!"

    train_ds = Subset(full_dataset, train_idx)
    val_ds = Subset(full_dataset, val_idx)
    dist = DistributedManager()

    def _loader(ds):
        sampler = InfiniteSampler(
            ds, rank=dist.rank, num_replicas=dist.world_size, seed=0
        )
        return DataLoader(
            ds, sampler=sampler, batch_size=batch_size, **DATA_LOADER_KWARGS
        )

    return train_idx, val_idx, _loader(train_ds), _loader(val_ds)

Multi-lead (24 h) dataset from the config

In [ ]:
cfg = OmegaConf.load(
    f"{CONF_ROOT}/config_training_aifs_single_lt_aware_FiLM_Time_regression_STEP1.yaml"
)

full_dataset = init_data(cfg)
train_idx, val_idx, train_loader, val_loader = build_weekly_split_loaders(
    full_dataset, batch_size=256
)

Quick batch shape check

In [ ]:
batch = next(iter(val_loader))
y_t, x_t, l, time = batch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"x_t: {x_t.shape}, y_t: {y_t.shape}, leads: {l.shape}, time: {time.shape}")

## 2) Define and evaluate Baselines

In [ ]:
# build train/test data for baselies
baseline_data = build_baseline_spatiotemporal_from_dataset(
    base_dataset=full_dataset,
    train_idx=train_idx,
    val_idx=val_idx,
    precip_out_channel=0,
)

feats_tr = baseline_data["feats_train"]
target_tr = baseline_data["target_train"]
feats_va = baseline_data["feats_val"]
target_va = baseline_data["target_val"]
lead_levels = baseline_data["lead_levels"]
meta = baseline_data["meta"]

aifs_tr = feats_tr[..., 0]  # AIFS low-res precip (first input channel)
aifs_va = feats_va[..., 0]
print(f"feats_tr: {feats_tr.shape}, target_tr: {target_tr.shape}")
print(f"feats_va: {feats_va.shape}, target_va: {target_va.shape}")

In [ ]:
def mse_all(y_true, y_pred):
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    return float(np.mean((y_true[mask] - y_pred[mask]) ** 2)) if mask.any() else np.nan


# set up baselines
BASELINES = [
    ("B0_raw", BaselineB0_Raw(), "aifs"),
    ("B1_affine_global", BaselineB1_AffineGlobal(), "aifs"),
    ("B2_affine_perLead", BaselineB2_AffinePerLeadGlobal(), "aifs"),
    ("B3_affine_per_grid", BaselineB3_AffinePerGrid(eps=1e-8), "aifs"),
    ("B4_multi_linear", BaselineB4_MultiLinearPerGrid(l2_reg=1e-4), "feats"),
    ("B5_qm_per_grid", BaselineB5_QuantileMapping(min_samples=30), "aifs"),
    ("B7_ridge_global", BaselineB7_RidgeGlobal(alpha=1.0), "feats"),
]

baseline_preds = {}
baseline_results = {}

# train and evaluate baselines
for name, model, input_type in BASELINES:
    X_tr = feats_tr if input_type == "feats" else aifs_tr
    X_va = feats_va if input_type == "feats" else aifs_va
    model.fit(X_tr, target_tr)
    pred = model.predict(X_va)
    baseline_preds[name] = pred
    baseline_results[name] = mse_all(target_va, pred)
    print(f"{name}: val MSE = {baseline_results[name]:.6f}")

To adapt baseline B4 (convert to xarray) for the diffusion pipeline (B4 is used as baseline in the super resolution)

In [ ]:
example_bc = xr.open_dataset(
    f"{DATA_ROOT}/AIFS_bias_corrected_precip_VAL_WeeklySplit_UNet_NoCurri_lam2.nc"
)

pred_b4_va_ds = xr.Dataset(
    data_vars={
        "precip6h_cumulative_capped": (
            ["time", "prediction_delta", "lat", "lon"],
            baseline_preds["B4_multi_linear"],
        )
    },
    coords={
        "time": pd.to_datetime(example_bc.time),
        "prediction_delta": pd.to_timedelta(example_bc.prediction_delta),
        "lat": example_bc.lat,
        "lon": example_bc.lon,
    },
)
pred_b4_va_ds.to_netcdf(
    f"{DATA_ROOT}/AIFS_bias_corrected_precip_VAL_WeeklySplit_BaselineB4.nc"
)

Extract per-lead metrics for the baselines

In [ ]:
per_lead_baselines = {}
for name, pred in [
    ("B0", baseline_preds["B0_raw"]),
    ("B1", baseline_preds["B1_affine_global"]),
    ("B2", baseline_preds["B2_affine_perLead"]),
    ("B3", baseline_preds["B3_affine_per_grid"]),
    ("B4", baseline_preds["B4_multi_linear"]),
    ("B5", baseline_preds["B5_qm_per_grid"]),
    ("B7", baseline_preds["B7_ridge_global"]),
]:
    per_lead_baselines[name] = baseline_per_lead_metrics(
        target_TLHW=target_va,
        pred_TLHW=pred,
        lead_levels=lead_levels,
        compute_spatial=True,
    )

## 3) Load trained deep learning bias-correction models

The 13 checkpoints below correspond to the following paper variants:

| # | Paper name | Description |
|---|---|---|
| 1 | UNet λ=0, Curri 8 leads/ep | Curriculum learning, no temporal consistency |
| 2 | UNet λ=0, No Curri | Plain training, no temporal consistency |
| 3 | **UNet λ=2 (final model)** | Final Step 1 model used in the paper |
| 4 | UNet MultiFT | Multi-stage fine-tuned variant |
| 5 | UNet λ=2, Curri | Combined ablation |
| 6-9 | UNet 12-horizon variants | Sub-range models (early/late lead times) |
| 10 | UNet Single 6h | Specialist baseline |
| 11 | UNet Single 3day | Specialist baseline |
| 12 | UNet Single 6day | Specialist baseline |
| 13 | Strategy 2 (3-day) | Direct regression ablation |

If you only want to load the final model (variant #3), comment out the other
checkpoint loads. To use the checkpoints from the paper, contact the corresponding author for access (see README). Otherwise, retrain using the configs in ../conf/ and update the paths below to point to your local checkpoint files.

In [ ]:
# -------- checkpoint dir for the mode to be tested --------
# UNet 24 horizons models
checkpoint_dir1 = f"{OUT_ROOT}/aifs_single_STEP1_lt_aware_24horizons_weeklySplit_FiLM_CoherentTime_attn16_8_4_Curriculum_GOOD_8leadsPerEp2_lam0_b1024_HuberLoss_UNet_FiLMemb64_mlp256/checkpoints_lt_aware_FiLM_regression_bias_correction"
checkpoint_dir2 = f"{OUT_ROOT}/aifs_single_STEP1_lt_aware_weeklySplit_FiLM_CoherentTime_attn16_8_4_lam0_b1024_HuberLoss_UNet_FiLMemb64_mlp256_TEST/checkpoints_lt_aware_FiLM_regression_bias_correction"
checkpoint_dir3 = f"{OUT_ROOT}/aifs_single_STEP1_lt_aware_weeklySplit_FiLM_CoherentTime_attn16_8_4_b1024_HuberLoss_UNet_FiLMemb64_mlp256_lam2/checkpoints_lt_aware_FiLM_regression_bias_correction"
checkpoint_dir4 = f"{OUT_ROOT}/aifs_single_STEP1_lt_aware_weeklySplit_FiLM_CoherentTime_attn16_8_4_lam0_b1024_HuberLoss_UNet_FiLMemb64_mlp256_MultiStepFineTuning_2/checkpoints_lt_aware_FiLM_regression_bias_correction"
checkpoint_dir5 = f"{OUT_ROOT}/aifs_single_STEP1_lt_aware_weeklySplit_FiLM_CoherentTime_attn16_8_4_b1024_HuberLoss_UNet_FiLMemb64_mlp256_lam2_Curri8leadsPerStep/checkpoints_lt_aware_FiLM_regression_bias_correction"

##### UNet 12 horizons models
# 0-12, Curri 4 leads, no lam
checkpoint_dir6 = f"{OUT_ROOT}/aifs_single_STEP1_lt_aware_weeklySplit_FiLM_CoherentTime_attn16_8_4_lam0_b1024_HuberLoss_UNet_FiLMemb64_mlp256_12horizons/checkpoints_lt_aware_FiLM_regression_bias_correction"
# 0-12, No Curri, lam2
checkpoint_dir7 = f"{OUT_ROOT}/aifs_single_STEP1_lt_aware_weeklySplit_FiLM_CoherentTime_attn16_8_4_b1024_HuberLoss_UNet_FiLMemb64_mlp256_12Horizons_NoCurri_lam2/checkpoints_lt_aware_FiLM_regression_bias_correction"
# 12-24, Curri 4 leads, no lam
checkpoint_dir8 = f"{OUT_ROOT}/aifs_single_STEP1_lt_aware_weeklySplit_FiLM_CoherentTime_attn16_8_4_b1024_HuberLoss_UNet_FiLMemb64_mlp256_12_24Horizons_lam0_Curri4leadsPerEp/checkpoints_lt_aware_FiLM_regression_bias_correction"
# 12-24, No Curri, lam2
checkpoint_dir9 = f"{OUT_ROOT}/aifs_single_STEP1_lt_aware_weeklySplit_FiLM_CoherentTime_attn16_8_4_b1024_HuberLoss_UNet_FiLMemb64_mlp256_12_24Horizons_NoCurri_lam2/checkpoints_lt_aware_FiLM_regression_bias_correction"

##### UNet single horizons models
# 6h
checkpoint_dir10 = f"{OUT_ROOT}/aifs_single_STEP1_weeklySplit_attn16_8_4_b1024_UNet_singleHori_6h_2/checkpoints_regression_bias_correction"
# 3day
checkpoint_dir11 = f"{OUT_ROOT}/aifs_single_STEP1_weeklySplit_attn16_8_4_b1024_UNet_singleHori_3day/checkpoints_regression_bias_correction"
# 6day
checkpoint_dir12 = f"{OUT_ROOT}/aifs_single_STEP1_weeklySplit_attn16_8_4_b1024_UNet_singleHori_6day/checkpoints_regression_bias_correction"

##### Direct strategy 2, single lead 3 days model!
checkpoint_dir13 = (
    f"{OUT_ROOT}/aifs_single_reg_DirectStrat2_3day_long/checkpoints_regression"
)


# 1) Find your .mdlus model file in the checkpoint directory
mdl_files1 = [f for f in os.listdir(checkpoint_dir1) if f.endswith(".mdlus")]
mdl_files2 = [f for f in os.listdir(checkpoint_dir2) if f.endswith(".mdlus")]
mdl_files3 = [f for f in os.listdir(checkpoint_dir3) if f.endswith(".mdlus")]
mdl_files4 = [f for f in os.listdir(checkpoint_dir4) if f.endswith(".mdlus")]
mdl_files5 = [f for f in os.listdir(checkpoint_dir5) if f.endswith(".mdlus")]

mdl_files6 = [f for f in os.listdir(checkpoint_dir6) if f.endswith(".mdlus")]
mdl_files7 = [f for f in os.listdir(checkpoint_dir7) if f.endswith(".mdlus")]
mdl_files8 = [f for f in os.listdir(checkpoint_dir8) if f.endswith(".mdlus")]
mdl_files9 = [f for f in os.listdir(checkpoint_dir9) if f.endswith(".mdlus")]

mdl_files10 = [f for f in os.listdir(checkpoint_dir10) if f.endswith(".mdlus")]
mdl_files11 = [f for f in os.listdir(checkpoint_dir11) if f.endswith(".mdlus")]
mdl_files12 = [f for f in os.listdir(checkpoint_dir12) if f.endswith(".mdlus")]

mdl_files13 = [f for f in os.listdir(checkpoint_dir13) if f.endswith(".mdlus")]

print("Model files:", mdl_files1)
print("Model files:", mdl_files2)
print("Model files:", mdl_files3)
print("Model files:", mdl_files4)
print("Model files:", mdl_files5)

# 2) Pick the one model we want
mdl_path1 = os.path.join(checkpoint_dir1, "UNetResidualCorrectorClampedFiLM.0.3.mdlus")
mdl_path2 = os.path.join(
    checkpoint_dir2, "UNetResidualCorrectorClampedFiLM.0.539648.mdlus"
)
mdl_path3 = os.path.join(
    checkpoint_dir3, "UNetResidualCorrectorClampedFiLM.0.134144.mdlus"
)
mdl_path4 = os.path.join(
    checkpoint_dir4, "UNetResidualCorrectorClampedFiLM.0.3648512.mdlus"
)
mdl_path5 = os.path.join(
    checkpoint_dir5, "UNetResidualCorrectorClampedFiLM.0.201728.mdlus"
)

mdl_path6 = os.path.join(checkpoint_dir6, "UNetResidualCorrectorClampedFiLM.0.9.mdlus")
mdl_path7 = os.path.join(
    checkpoint_dir7, "UNetResidualCorrectorClampedFiLM.0.404480.mdlus"
)
mdl_path8 = os.path.join(
    checkpoint_dir8, "UNetResidualCorrectorClampedFiLM.0.438272.mdlus"
)
mdl_path9 = os.path.join(
    checkpoint_dir9, "UNetResidualCorrectorClampedFiLM.0.607232.mdlus"
)

mdl_path10 = os.path.join(checkpoint_dir10, "UNet.0.50432.mdlus")
mdl_path11 = os.path.join(checkpoint_dir11, "UNet.0.50432.mdlus")
mdl_path12 = os.path.join(checkpoint_dir12, "UNet.0.58880.mdlus")

mdl_path13 = os.path.join(checkpoint_dir13, "UNet.0.40.mdlus")

# 3) Load it via PhysicsNeMo’s Module.from_checkpoint
model1 = Module.from_checkpoint(mdl_path1, {})
model2 = Module.from_checkpoint(mdl_path2, {})
model3 = Module.from_checkpoint(mdl_path3, {})
model4 = Module.from_checkpoint(mdl_path4, {})
model5 = Module.from_checkpoint(mdl_path5, {})

model6 = Module.from_checkpoint(mdl_path6, {})
model7 = Module.from_checkpoint(mdl_path7, {})
model8 = Module.from_checkpoint(mdl_path8, {})
model9 = Module.from_checkpoint(mdl_path9, {})

model10 = Module.from_checkpoint(mdl_path10, {})
model11 = Module.from_checkpoint(mdl_path11, {})
model12 = Module.from_checkpoint(mdl_path12, {})

model13 = Module.from_checkpoint(mdl_path13, {})

# 4) Move to device and eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model1 = model1.to(device).eval()
model2 = model2.to(device).eval()
model3 = model3.to(device).eval()
model4 = model4.to(device).eval()
model5 = model5.to(device).eval()

model6 = model6.to(device).eval()
model7 = model7.to(device).eval()
model8 = model8.to(device).eval()
model9 = model9.to(device).eval()

model10 = model10.to(device).eval()
model11 = model11.to(device).eval()
model12 = model12.to(device).eval()

model13 = model13.to(device).eval()

Load ViT model from checkpoint

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- 1) Rebuild the ViT with the same hyperparams as in training ----
# Base info from dataset
in_ch = len(full_dataset.input_channels())
out_ch = len(full_dataset.output_channels())
img_shape = full_dataset.image_shape()  # e.g. (16, 32)

# vit_args = getattr(cfg.model, "model_args", {})

vit_model = ViTBiasCorrectorFiLM(
    img_in_channels=in_ch,
    img_out_channels=out_ch,
    img_resolution=img_shape,
    patch_size=4,
    d_model=128,
    depth=6,
    n_heads=4,
    mlp_hidden=256,
    n_leads=24,
    emb_dim=64,
).to(device)

# ---- 2) Load checkpoint ----
ckpt_path = f"{OUT_ROOT}/aifs_single_STEP1_lt_aware_weeklySplit_FiLM_CoherentTime_attn16_8_4_b1024_HuberLoss_FiLMemb64_mlp256_ViT_all_24Horizons_lam2_NoCurri/checkpoints_lt_aware_FiLM_regression_bias_correction/ViTBiasCorrectorFiLM.0.4594688.pt"

ckpt = torch.load(ckpt_path, map_location=device)
if "state_dict" in ckpt:
    state_dict = ckpt["state_dict"]
else:
    state_dict = ckpt

# If you trained under DDP, keys may be 'module.xxx' – strip "module." if needed:
if any(k.startswith("module.") for k in state_dict.keys()):
    state_dict = {k.replace("module.", "", 1): v for k, v in state_dict.items()}

missing, unexpected = vit_model.load_state_dict(state_dict, strict=False)
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

vit_model.eval()


## 4) Visualise predictions

In [ ]:
fig = Visualize_Pred_onebatch_ByLeadTime3(
    [model2, model3, model4, vit_model],
    ["UNet$_{\\lambda=0}$", "UNet$_{\\lambda=2}$", "Multi FT", "ViT"],
    val_loader=val_loader,
    full_dataset=full_dataset,
    cfg=cfg,
    lead_to_plot=[2, 11],
    sample_indices={2: [3, 4, 5], 11: [1, 2, 5]},
)
# fig.savefig(f"{RESULTS_ROOT}/BC_images_lead2_and_11.png", dpi=300, bbox_inches="tight")

## 5) Evaluate models

Compute metrics for aggregated metrics (all leads averaged)

In [ ]:
MULTI_LEAD_MODELS = [
    ("model1", model1),
    ("model2", model2),
    ("model3", model3),
    ("model4", model4),
    ("model5", model5),
    ("model6", model6),
    ("model7", model7),
    ("model8", model8),
    ("model9", model9),
]

eval_results = {}
for name, m in MULTI_LEAD_MODELS:
    eval_results[name] = EvaluateModel(
        m,
        train_loader,
        val_loader,
        lt_aware=True,
        compute_spatial=True,
    )

Single-horizon datasets and loaders

> **Note:** `cfg_3day` is shared between the single 3-day bias-correction model and the
> direct strategy-2 model, but they differ on whether altitude is included.
> Reload the config with/without altitude as needed.

In [ ]:
cfg_6h = OmegaConf.load(
    f"{CONF_ROOT}/config_training_aifs_single_regression_STEP1_6h.yaml"
)
cfg_3day = OmegaConf.load(
    f"{CONF_ROOT}/config_training_aifs_single_regression_STEP1_3day.yaml"
)
cfg_6day = OmegaConf.load(
    f"{CONF_ROOT}/config_training_aifs_single_regression_STEP1_6day.yaml"
)

full_dataset_6h = init_data(cfg_6h)
full_dataset_3day = init_data(cfg_3day)
full_dataset_6day = init_data(cfg_6day)

_, _, train_loader_6h, val_loader_6h = build_weekly_split_loaders(full_dataset_6h)
_, _, train_loader_3day, val_loader_3day = build_weekly_split_loaders(full_dataset_3day)
_, _, train_loader_6day, val_loader_6day = build_weekly_split_loaders(full_dataset_6day)

In [ ]:
SINGLE_LEAD_MODELS = [
    ("model10", model10, train_loader_6h, val_loader_6h),
    ("model11", model11, train_loader_3day, val_loader_3day),
    ("model12", model12, train_loader_6day, val_loader_6day),
]

for name, m, tr_l, va_l in SINGLE_LEAD_MODELS:
    eval_results[name] = EvaluateModel(
        m, tr_l, va_l, lt_aware=False, compute_spatial=True
    )

Direct strategy-2 3-day model (high-res with coarsening)

In [ ]:
# Reload config with altitude for strategy-2 model
cfg_3day = OmegaConf.load(
    f"{CONF_ROOT}/config_training_aifs_single_regression_STEP1_3day.yaml"
)
full_dataset_3day = init_data(cfg_3day)
_, _, train_loader_3day, val_loader_3day = build_weekly_split_loaders(
    full_dataset_3day, batch_size=8
)

HIGH_RES_SHAPE = (608, 640)

eval_results["model13"] = EvaluateModel(
    model13,
    train_loader_3day,
    val_loader_3day,
    lt_aware=False,
    compute_spatial=True,
    coarsen=True,
    coarsen_mode="avg",
    upsample_input=True,
    upsample_shape=HIGH_RES_SHAPE,
    upsample_mode="bilinear",
)

Per-lead-time evaluation

In [ ]:
results_per_lead = {}

for name, m in MULTI_LEAD_MODELS:
    results_per_lead[name] = EvaluateModelPerLead(
        m,
        train_loader,
        val_loader,
        lt_aware=True,
        compute_spatial=True,
    )

results_per_lead["vit"] = EvaluateModelPerLead(
    vit_model,
    train_loader,
    val_loader,
    lt_aware=True,
    compute_spatial=True,
)

In [ ]:
for name, m, tr_l, va_l in SINGLE_LEAD_MODELS:
    results_per_lead[name] = EvaluateModelPerLead(
        m,
        tr_l,
        va_l,
        lt_aware=False,
        compute_spatial=True,
    )

In [ ]:
# Strategy-2 3-day model
cfg_3day = OmegaConf.load(
    f"{CONF_ROOT}/config_training_aifs_single_regression_STEP1_3day.yaml"
)
full_dataset_3day = init_data(cfg_3day)
_, _, train_loader_3day, val_loader_3day = build_weekly_split_loaders(
    full_dataset_3day, batch_size=2
)

results_per_lead["model13"] = EvaluateModelPerLead(
    model13,
    train_loader_3day,
    val_loader_3day,
    lt_aware=False,
    compute_spatial=True,
    coarsen=True,
    coarsen_mode="avg",
    upsample_input=True,
    upsample_shape=HIGH_RES_SHAPE,
    upsample_mode="bilinear",
)

Save per-lead results

In [ ]:
# Save all results
for name, data in results_per_lead.items():
    with open(f"results_PerLead_{name}.pickle", "wb") as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

## 6) Produce results tables/figures

Load models metrics

In [ ]:
# Load all results
MODEL_NAMES = [
    "model1",
    "model2",
    "model3",
    "model4",
    "model5",
    "model6",
    "model7",
    "model8",
    "model9",
    "model10",
    "model11",
    "model12",
    "model13",
    "vit",
]
results_per_lead = {}
for name in MODEL_NAMES:
    with open(f"{RESULTS_ROOT}/results_PerLead_{name}.pickle", "rb") as f:
        results_per_lead[name] = pickle.load(f)

In [ ]:
baselines = {
    "B0: AIFS": per_lead_baselines["B0"],
    "B1: Global Affine": per_lead_baselines["B1"],
    "B2: Global Affine/Lead": per_lead_baselines["B2"],
    "B3: Pixel LR, Precip": per_lead_baselines["B3"],
    "B4: Pixel LR, MultiVar": per_lead_baselines["B4"],
    "B5: Quantile Mapping": per_lead_baselines["B5"],
    "B7: Ridge Reg Global": per_lead_baselines["B7"],
}

multi_lead_models = {
    "UNet, Curri, no λ": results_per_lead["model1"],
    "UNet, No Curri, no λ": results_per_lead["model2"],
    "UNet, No Curri, λ=2": results_per_lead["model3"],
    "UNet Multi FineTuned": results_per_lead["model4"],
    "UNet, Curri + λ=1": results_per_lead["model5"],
    "ViT": results_per_lead["vit"],
}

single_lead_models = {
    "UNet Single 6h": (0, results_per_lead["model10"]),
    "UNet Single 3day": (11, results_per_lead["model11"]),
    "UNet Single 6day": (23, results_per_lead["model12"]),
    "UNet Strat2 3day": (11, results_per_lead["model13"]),
}

Metric comparison plots

In [ ]:
PLOT_KWARGS = dict(
    baselines=baselines,
    multi_lead_models=multi_lead_models,
    single_lead_models=single_lead_models,
    split="val",
)

for metric, ylim in [
    ("MSE_all", (0, 0.06)),
    ("WeightedMSE", (0.02, 0.11)),
    ("MSE_wet", (0.04, 0.18)),
    ("precision", (0.2, 0.8)),
    ("recall_POD", (0.3, 1)),
]:
    fig, ax = plot_metric_comparison_grouped_legend(
        metric=metric,
        ylim=ylim,
        save_path=f"{RESULTS_ROOT}/Results_Plot_BiasCorrection_ByLead_{metric}_LAST.png",
        title=f"Precipitation Bias Correction: {metric} by Lead Time",
        **PLOT_KWARGS,
    )

In [ ]:
fig_fss, _ = plot_fss_by_lead(
    **PLOT_KWARGS,
    neighborhoods=[1, 5, 17],
    suptitle="Fractions Skill Score by Lead Time",
    save_path=f"{RESULTS_ROOT}/FSS_by_lead.png",
)

fig_sal, _ = plot_sal_by_lead(
    **PLOT_KWARGS,
    suptitle="SAL Decomposition by Lead Time",
    save_path=f"{RESULTS_ROOT}/SAL_by_lead.png",
)

Generate LaTeX tables

In [ ]:
TABLE_KWARGS = dict(
    baselines=baselines,
    multi_lead_models=multi_lead_models,
    single_lead_models=single_lead_models,
)
LEADS = [0, 1, 2, 3, 7, 11, 15, 19, 23]

for metric, caption in [
    ("CSI", "CSI by lead time"),
    ("MSE_wet", "MSE_wet by lead time"),
    ("FSS_5", "FSS_5 by lead time"),
]:
    latex, df = generate_per_lead_latex_table(
        **TABLE_KWARGS,
        metric=metric,
        leads_to_show=LEADS,
        caption=caption,
        save_path=f"{RESULTS_ROOT}/STEP1_table_{metric}_byLead.tex",
    )

latex_compact, df_compact = generate_compact_latex_table(
    **TABLE_KWARGS,
    metrics=["MSE_all", "MSE_wet", "CSI", "FSS_5", "SAL_S", "SAL_A", "SAL_L"],
    caption="Model comparison (average across all leads)",
    save_path=f"{RESULTS_ROOT}/table_compact.tex",
)

latex_full, df_per_lead, df_avg = generate_latex_tables(
    **TABLE_KWARGS,
    metrics=["MSE_all", "CSI", "FSS_5"],
    leads_to_show=[0, 11, 23],
    caption="Precipitation bias correction results",
    save_path=f"{RESULTS_ROOT}/tables_full.tex",
)

## 7) To run bias correction inference (without the super resolution step)

Run bias-correction inference over the full validation set and save results.

Multi-lead models

In [ ]:
# UNet No Curri, λ=2
RunInferenceWeekly(
    CONF_ROOT,
    checkpoint_dir3,
    os.path.join(checkpoint_dir3, "UNetResidualCorrectorClampedFiLM.0.134144.mdlus"),
    f"{DATA_ROOT}/AIFS_bias_corrected_precip_VAL_WeeklySplit_UNet_NoCurri_lam2.nc",
)

# MultiStepFineTuning_2
RunInferenceWeekly(
    CONF_ROOT,
    checkpoint_dir4,
    os.path.join(checkpoint_dir4, "UNetResidualCorrectorClampedFiLM.0.3648512.mdlus"),
    f"{DATA_ROOT}/AIFS_bias_corrected_precip_VAL_WeeklySplit_UNet_MultiStepFineTuning_2.nc",
)

Single-lead models

In [ ]:
SINGLE_INFERENCES = [
    ("6h", cfg_6h, checkpoint_dir10, "UNet.0.50432.mdlus", np.timedelta64(6, "h")),
    ("3day", cfg_3day, checkpoint_dir11, "UNet.0.50432.mdlus", np.timedelta64(3, "D")),
    ("6day", cfg_6day, checkpoint_dir12, "UNet.0.58880.mdlus", np.timedelta64(6, "D")),
]

for label, cfg_i, ckpt_dir, mdl_name, delta_val in SINGLE_INFERENCES:
    cfg_path_i = (
        f"{CONF_ROOT}/config_training_aifs_single_regression_STEP1_{label}.yaml"
    )
    RunInferenceWeeklySingleLead(
        cfg_path=cfg_path_i,
        checkpoint_dir=ckpt_dir,
        mdl_path=os.path.join(ckpt_dir, mdl_name),
        out_path=f"{DATA_ROOT}/AIFS_bias_corrected_precip_VAL_WeeklySplit_UNet_SingleLead_{label}.nc",
        batch_size=256,
        data_loader_kwargs=DATA_LOADER_KWARGS,
        add_prediction_delta_dim=True,
        prediction_delta_value=delta_val,
    )

Post-processing: interpolate bias-corrected data to SR grid

Interpolate the low-res bias-corrected output onto the high-resolution grid
used by the diffusion model (step 2), then save as NetCDF.

In [ ]:
# Load the SR (diffusion) dataset config to get the high-res lat/lon grid
cfg_step2 = OmegaConf.load(f"{CONF_ROOT}/config_training_aifs_single_diffusion.yaml")

dataset_cfg_step2 = OmegaConf.to_container(cfg_step2.dataset, resolve=True)
register_dataset(cfg_step2.dataset.type)
sr_dataset, _ = init_dataset_from_config(
    dataset_cfg_step2, {}, None, cfg_step2.training.hp.seed
)

lat_hr = sr_dataset.latitude()
lon_hr = sr_dataset.longitude()
print(f"SR grid: {len(lat_hr)} x {len(lon_hr)}")

In [ ]:
def interpolate_bc_to_sr_grid(ds_bc_path, zarr_store, netcdf_out, lat_hr, lon_hr):
    """Interpolate bias-corrected data to SR grid, save as zarr then convert to NetCDF."""
    import shutil

    ds_bc = xr.open_dataset(ds_bc_path)
    var = "precip6h_cumulative_capped"

    # Filter out all-NaN time steps
    valid_times = ~ds_bc[var].isnull().all(dim=("prediction_delta", "lat", "lon"))
    ds_bc = ds_bc.sel(time=ds_bc.time[valid_times])

    if os.path.exists(zarr_store):
        shutil.rmtree(zarr_store)

    for i in range(ds_bc.sizes["prediction_delta"]):
        print(f"  horizon {i}")
        ds_h = ds_bc.isel(prediction_delta=slice(i, i + 1))
        ds_h_hr = ds_h.interp(lat=lat_hr, lon=lon_hr, method="linear")
        ds_h_hr = ds_h_hr.chunk(
            {"time": 64, "lat": 256, "lon": 256, "prediction_delta": 1}
        )
        ds_h_hr.to_zarr(
            zarr_store, mode="w" if i == 0 else "a", append_dim="prediction_delta"
        )

    # Convert zarr → NetCDF
    ds_hr = xr.open_zarr(zarr_store)
    ds_hr.to_netcdf(
        netcdf_out,
        engine="netcdf4",
        encoding={var: {"zlib": True, "complevel": 4, "chunksizes": (1, 1, 256, 256)}},
    )
    print(f"  → saved {netcdf_out}")

Visual sanity checks on interpolated bias-corrected data

In [ ]:
bc_path = f"{DATA_ROOT}/AIFS_bias_corrected_precip_VAL_WeeklySplit_UNet_MultiStepFineTuning_2_AllHori_interp_good.nc"
bc_data = xr.open_dataset(bc_path)

fig, axes = plt.subplots(6, 4, figsize=(15, 15))
for i, (ax, hori) in enumerate(zip(axes.flat, bc_data.prediction_delta)):
    bc_data.precip6h_cumulative_capped[135].sel(prediction_delta=hori).plot(ax=ax)
plt.tight_layout()
plt.show()